In [1]:
%uv pip install jiwer
%uv pip install datasets
%uv pip install torch==2.8 
%uv pip install torchcodec==0.7
%uv pip install speechbrain

Using Python 3.12.6 environment at: /usr/local
Resolved 3 packages in 131ms
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
jiwer      ------------------------------     0 B/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Preparing packages... (0/2)
jiwer      ------------------------------ 14.95 KiB/22.49 KiB
⠙ Prepar

In [3]:
import re
import numpy as np
import torch
from datasets import load_dataset
from jiwer import wer, cer
from tqdm import tqdm
from speechbrain.inference.ASR import EncoderASR

# =============================================================================
# Arabic Text Normalization
# =============================================================================
def normalize_arabic_text(text):
    """Your existing normalization function (unchanged)"""
    punctuation = r'[!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~،؛؟]'
    text = re.sub(punctuation, '', text)
    diacritics = r'[\u064B-\u0652]'
    text = re.sub(diacritics, '', text)
    text = re.sub('پ', 'ب', text)
    text = re.sub('ڤ', 'ف', text)
    text = re.sub(r'[آ]', 'ا', text)
    text = re.sub(r'[أإ]', 'ا', text)
    text = re.sub(r'[ؤ]', 'و', text)
    text = re.sub(r'[ئ]', 'ي', text)
    text = re.sub(r'[ء]', '', text)
    eastern_to_western_numerals = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
    }
    for eastern, western in eastern_to_western_numerals.items():
        text = text.replace(eastern, western)
    return text.strip()

# =============================================================================
# SpeechBrain Model Evaluation Function - FIXED VERSION
# =============================================================================
def evaluate_speechbrain(
    model_name, 
    dataset_config, 
    max_samples=50,
    batch_size=4,
    device="cuda"
):
    print(f"\nEvaluating {model_name}...")
    print(f"Dataset: {dataset_config['name']} ({dataset_config['split']} split)")
    
    try:
        # Load SpeechBrain EncoderASR model
        asr_model = EncoderASR.from_hparams(
            source=model_name,
            savedir=f"pretrained_models/{model_name.split('/')[-1]}",
            run_opts={"device": device}
        )
        
        # Load dataset
        dataset = load_dataset(dataset_config["name"], split=dataset_config["split"])
        
        # Limit samples if specified
        if max_samples is not None and len(dataset) > max_samples:
            dataset = dataset.select(range(max_samples))
            print(f"Limited to {max_samples} samples")
        
        references = []
        predictions = []
        
        audio_col = dataset_config["audio_col"]
        text_col = dataset_config["text_col"]
        
        for i in tqdm(range(0, len(dataset), batch_size), desc="Processing batches"):
            batch = dataset[i:i+batch_size]
            
            # Extract audio and text
            audio_items = batch[audio_col]
            reference_texts = batch[text_col]
            
            # Prepare audio tensors with proper padding
            wavs = []
            lengths = []
            
            for audio_item in audio_items:
                audio_array = audio_item["array"]
                
                # Convert to torch tensor
                if isinstance(audio_array, np.ndarray):
                    audio_tensor = torch.from_numpy(audio_array)
                else:
                    audio_tensor = torch.FloatTensor(audio_array)
                
                # Ensure float32
                audio_tensor = audio_tensor.float()
                
                # Handle stereo audio (take first channel)
                if audio_tensor.dim() == 2:
                    audio_tensor = audio_tensor[0]
                
                # Remove any extra dimensions
                audio_tensor = audio_tensor.squeeze()
                
                # Store tensor and its original length
                wavs.append(audio_tensor)
                lengths.append(audio_tensor.shape[0])
            
            # Find max length in batch
            max_len = max(lengths)
            
            # Create padded batch tensor [batch, time]
            padded_wavs = []
            for wav in wavs:
                if wav.shape[0] < max_len:
                    # Pad with zeros (SpeechBrain expects padding at the end)
                    padding = torch.zeros(max_len - wav.shape[0], dtype=wav.dtype)
                    wav = torch.cat([wav, padding])
                padded_wavs.append(wav)
            
            # Stack padded tensors into batch
            wavs_batch = torch.stack(padded_wavs)
            
            # Create relative lengths tensor [0.0, 1.0]
            wav_lens = torch.tensor([l / max_len for l in lengths], dtype=torch.float32)
            
            # Move to device
            if device == "cuda":
                wavs_batch = wavs_batch.cuda()
                wav_lens = wav_lens.cuda()
            
            # Transcribe batch
            with torch.no_grad():
                transcriptions = asr_model.transcribe_batch(wavs_batch, wav_lens)[0]
            
            # Normalize and store results
            for ref, pred in zip(reference_texts, transcriptions):
                references.append(normalize_arabic_text(ref))
                predictions.append(normalize_arabic_text(pred))
        
        # Calculate metrics
        wer_score = wer(references, predictions)
        cer_score = cer(references, predictions)
        
        # Calculate accuracy
        exact_matches = sum(1 for ref, pred in zip(references, predictions) if ref.strip() == pred.strip())
        accuracy = exact_matches / len(references)
        
        return {
            "model": model_name,
            "dataset": dataset_config["name"],
            "wer": wer_score,
            "cer": cer_score,
            "accuracy": accuracy,
            "samples_tested": len(references),
            "references": references,
            "predictions": predictions
        }
    
    except Exception as e:
        print(f"Error during evaluation: {str(e)[:200]}...")
        import traceback
        traceback.print_exc()
        return None

# =============================================================================
# Dataset Configuration
# =============================================================================
DATASETS = {
    # "egyptian_asr": {
    #     "name": "MightyStudent/Egyptian-ASR-MGB-3",
    #     "split": "train", 
    #     "audio_col": "audio",
    #     "text_col": "sentence"
    # }
    # ,
    "arabic_egy": {
        "name": "MAdel121/arabic-egy-cleaned", 
        "split": "test",
        "audio_col": "audio", 
        "text_col": "text"
    }
}

# =============================================================================
# Main Evaluation Execution
# =============================================================================
def main():
    MODEL = "speechbrain/asr-wav2vec2-commonvoice-14-ar"
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    all_results = []
    
    for dataset_name, dataset_config in DATASETS.items():
        print(f"\n{'='*60}")
        print(f"DATASET: {dataset_name}")
        print(f"{'='*60}")
        
        result = evaluate_speechbrain(
            MODEL,
            dataset_config,
            max_samples=None,
            batch_size=1,
            device=device
        )
        
        if result is not None:
            all_results.append(result)
            print(f"\nResults:")
            print(f"  WER: {result['wer']:.3f}")
            print(f"  CER: {result['cer']:.3f}")
    
    # Print summary
    if all_results:
        print("\n" + "="*60)
        print("EVALUATION SUMMARY")
        print("="*60)
        for result in all_results:
            print(f"{result['dataset']}: WER={result['wer']:.3f}, CER={result['cer']:.3f}, Acc={result['accuracy']:.3f}")
        
        return all_results
    else:
        print("\nNo successful evaluations completed.")
        return None

# Run the evaluation
if __name__ == "__main__":
    results = main()

Using device: cuda

DATASET: arabic_egy

Evaluating speechbrain/asr-wav2vec2-commonvoice-14-ar...
Dataset: MAdel121/arabic-egy-cleaned (test split)


speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

... (37 outputs left out)

Generating validation split:   0%|          | 0/10361 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10361 [00:00<?, ? examples/s]

Processing batches: 100%|████████████████████████████████████████████████| 10361/10361 [08:04<00:00, 21.37it/s]



Results:
  WER: 0.936
  CER: 0.516

EVALUATION SUMMARY
MAdel121/arabic-egy-cleaned: WER=0.936, CER=0.516, Acc=0.000
